# STAT 764 · Meeting 3 — Generalization and resampling

**Tuesday, September 15 · Concept 20 min · Studio 40 min**

Thursday you built a pipeline that produces an honest number. Today: that
number is itself a random variable, and it is noisier than you think.

By the end you will have watched three models trade the lead back and forth
purely on the luck of the split — and you will know why "I held out 25% and
model B won" is not yet evidence.

In [ ]:
import pathlib
import sys

here = pathlib.Path.cwd()
root = next(p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists())
sys.path.insert(0, str(root / "course"))

from stat764 import load

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, RepeatedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

ames = load("ames.csv")

NUMERIC = ["Gr_Liv_Area", "Lot_Area", "Year_Built", "Overall_Qual",
           "Total_Bsmt_SF", "Garage_Cars"]
CATEGORICAL = ["Neighborhood", "Central_Air"]

X = ames[NUMERIC + CATEGORICAL]
y = ames["SalePrice"]


def make(model):
    """The M2 pipeline, with the estimator swapped in."""
    prep = ColumnTransformer([
        ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([("prep", prep), ("model", model)])


print(f"{len(X)} houses, {len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical predictors")

## 1. Two errors, and only one of them matters

**Training error** measures how well the model describes data it has already
seen. It can always be driven to zero by making the model flexible enough. It
is a measure of *memory*.

**Generalization error** is the expected error on a new case drawn from the
same process. It is the only quantity anyone cares about, and you can never
observe it directly — you can only estimate it.

Everything in this course is about estimating the second thing without fooling
yourself, which is harder than it sounds because the first thing is sitting
right there looking like an answer.

## 2. Flexibility: watch the two errors separate

A decision tree has a dial — how deep it is allowed to grow. Turn the dial up
and the model can carve the training data more finely. Watch what each error
does.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=764)

rows = []
for d in depths:
    fitted = make(DecisionTreeRegressor(max_depth=d, random_state=0)).fit(X_train, y_train)
    rows.append({
        "depth": d,
        "training": r2_score(y_train, fitted.predict(X_train)),
        "unseen": r2_score(y_test, fitted.predict(X_test)),
    })

curve = pd.DataFrame(rows)
best = curve.loc[curve["unseen"].idxmax()]
print(curve.round(3).to_string(index=False))
print(f"\nBest on unseen data: depth {best.depth:.0f}, R-squared {best.unseen:.3f}")
print(f"At depth 25 the model explains {curve.iloc[-1]['training']:.3f} of the training data "
      f"and {curve.iloc[-1]['unseen']:.3f} of the unseen data.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(curve["depth"], curve["training"], marker="o", label="training data")
ax.plot(curve["depth"], curve["unseen"], marker="s", label="unseen data")
ax.axvline(best.depth, color="grey", linestyle=":", linewidth=1)
ax.annotate(f"best on unseen\n(depth {best.depth:.0f})",
            xy=(best.depth, best.unseen), xytext=(best.depth + 3, best.unseen - 0.18),
            arrowprops=dict(arrowstyle="->", color="grey"), fontsize=9, color="grey")
ax.set_xlabel("maximum tree depth  (more flexible →)")
ax.set_ylabel("R-squared")
ax.set_title("Training error always improves. Generalization error does not.")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The training curve climbs to 1.000 and stays there. The unseen curve rises,
peaks, and then **falls** — past the peak, extra flexibility is spent
memorising noise that will not repeat.

This is the bias–variance trade-off, and it is the reason model selection is a
real problem rather than "pick the one with the best fit."

It is also exactly what happened to whoever fit a full tree on September 8 and
reported 1.000.

## 3. One split is a random number

Here is the part that catches people. You did the honest thing — you held data
back. But *which* 25% you held back was arbitrary, and the answer moves.

Three reasonable models, fifty different random splits. Count the wins.

In [ ]:
CONTENDERS = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=10),
    "Tree(d=6)": DecisionTreeRegressor(max_depth=6, random_state=0),
}

results = {name: [] for name in CONTENDERS}
winners = []

for seed in range(50):
    A, B, a, b = train_test_split(X, y, test_size=0.25, random_state=seed)
    scores = {name: r2_score(b, make(m).fit(A, a).predict(B))
              for name, m in CONTENDERS.items()}
    for name, s in scores.items():
        results[name].append(s)
    winners.append(max(scores, key=scores.get))

summary = pd.DataFrame({
    "wins (of 50)": pd.Series(winners).value_counts().reindex(CONTENDERS, fill_value=0),
    "mean R2": {k: np.mean(v) for k, v in results.items()},
    "worst split": {k: np.min(v) for k, v in results.items()},
    "best split": {k: np.max(v) for k, v in results.items()},
})
summary["spread"] = summary["best split"] - summary["worst split"]
print(summary.round(3).to_string())

between = summary["mean R2"].max() - summary["mean R2"].min()
within = summary["spread"].mean()
print(f"\n  difference between the models:  {between:.3f}")
print(f"  variation from the split alone: {within:.3f}   ({within/between:.0f}x larger)")

Read that last pair of numbers carefully.

The models differ from each other by about **0.01** in mean R-squared. The
same model, re-split, moves by about **0.13**. The noise you introduced by
picking one arbitrary split is an order of magnitude larger than the effect
you were trying to measure.

All three models won a substantial share of the fifty splits. If you had run
one split and announced the winner, you would have been reporting the seed.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([results[k] for k in CONTENDERS], tick_labels=list(CONTENDERS))
for i, k in enumerate(CONTENDERS, start=1):
    ax.scatter(np.random.default_rng(0).normal(i, 0.04, len(results[k])),
               results[k], s=12, alpha=0.4)
ax.set_ylabel("R-squared on the held-out 25%")
ax.set_title("Fifty splits of the same data, three models")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4. k-fold cross-validation

The fix is not a better split. It is to stop relying on any single one.

**k-fold CV** cuts the data into k parts, then fits k times — each part gets a
turn as the test set while the other k−1 train. Every observation is used for
training and for testing, just never at the same time. You get k estimates,
and their average is a far steadier number than any one of them.

The pipeline matters here more than ever: `cross_val_score` refits **every
step** on each training fold. Because your imputer and scaler live inside the
pipeline, they are refit k times too — which is the only correct thing to do,
and would be nearly impossible to remember to do by hand.

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=764)

print(f"{'model':<12}{'one split (seed 0)':>20}{'10-fold CV':>16}{'SD across folds':>18}")
for name, m in CONTENDERS.items():
    A, B, a, b = train_test_split(X, y, test_size=0.25, random_state=0)
    single = r2_score(b, make(m).fit(A, a).predict(B))
    folds = cross_val_score(make(m), X, y, cv=cv, scoring="r2")
    print(f"{name:<12}{single:>20.3f}{folds.mean():>16.3f}{folds.std():>18.3f}")

On seed 0 the tree looks clearly worst. Under 10-fold CV the three are
indistinguishable. **The single split defamed the tree** — and if seed 0 was
the split you happened to run, you would have dropped a perfectly good model
on the strength of one arbitrary partition.

## 5. Repeated CV, and reporting uncertainty

10-fold CV still depends on one random assignment into folds. Repeating it with
different assignments and averaging costs nothing but time, and it lets you put
an interval around the estimate instead of a bare number.

In [ ]:
rcv = RepeatedKFold(n_splits=10, n_repeats=5, random_state=764)
folds = cross_val_score(make(Ridge(alpha=10)), X, y, cv=rcv, scoring="r2")

print(f"  {len(folds)} estimates from 5 repeats of 10-fold CV")
print(f"  mean R-squared {folds.mean():.3f}, SD {folds.std():.3f}")
print(f"  middle 90% of fold estimates: "
      f"{np.percentile(folds, 5):.3f} to {np.percentile(folds, 95):.3f}")
print("\n  Report the mean. Never report it without the spread.")

---

## Studio — make the winner change

Working in your team. Choose **three** models you consider genuinely
competitive — a tree of some depth, ridge with a penalty you pick, OLS on a
different column set, k-nearest-neighbours, whatever you can argue for.

Then answer, with evidence:

1. Over 50 random splits, **how often does each of your models win?**
2. Try to find a single `random_state` that makes your *worst* model look
   best. Report the seed, or report that you searched 300 and found none —
   **both are results.** If you find one in seconds, the models were within
   split noise of each other and no single split could have told them apart.
   If you cannot find one, the gap is real, and you have just measured that
   it is larger than the noise.
3. Under 10-fold CV, is there still a winner? Is the gap larger than the SD
   across folds?
4. If you had to recommend one model to a client on this evidence, what would
   you say — and what would you refuse to say?

Question 2 is not a trick. Searching seeds until one flatters your preferred
model is what a selective report looks like from the inside — and noticing how
fast it works is why the capstone asks for your evaluation design *before* your
results.

In [ ]:
# YOUR CODE HERE

## Compare

Each team reports: your three models, the win counts, and the seed you found
for question 2.

1. Did any team find a model that won under *every* reasonable evaluation?
2. How different were the "cherry-picked seed" results across teams?
3. Someone reports a single held-out R-squared of 0.86. What is the minimum
   you need to know before you believe it means anything?

## Exit ticket

> **You have a single held-out estimate and a 10-fold CV estimate that
> disagree. Which do you trust, and what would change your mind?**

---

## Labs

**Lab 1 is due tonight, 11:59 pm.** Upload the notebook to Canvas.

**Lab 2 is assigned today, due Tuesday Sep 22, 11:59 pm** — `course/labs/lab02/`.
`git pull` first, then copy the notebook into `work/` before you edit it.

---

## Also in this neighbourhood

📗 **The bias–variance decomposition, algebraically.** ISLP §2.2.2. Today you
watched the two curves separate; the decomposition says exactly what the gap is
made of. Worth doing once with a pencil, and note that it is a statement about
*expected* test error averaged over training sets — which is why a single split
was such a poor guide.

📗 **Resamplers that encode structure.** `GroupKFold`, `StratifiedGroupKFold`,
`TimeSeriesSplit`. `KFold` assumes rows are exchangeable, and yours often are
not — repeat patients, repeat customers, anything with a date. Thursday shows
what it costs to get this wrong; the fix is choosing the right splitter.

🚫 **AIC, BIC, and Mallows' Cp for model selection.** Not broken, and worth
understanding — BIC comes back in Week 13 to choose the number of clusters.

The difference is what they estimate. Information criteria estimate the
*optimism* of the training fit, analytically, using a penalty derived under model
assumptions — a likelihood, a correctly specified family, a parameter count that
means something. Cross-validation estimates out-of-sample error **directly**, by
holding data out, and needs none of that.

When the assumptions hold and n is small, information criteria can be more
efficient — they use every observation for fitting. When you are comparing a
random forest to a boosted tree, "number of parameters" is not even well defined,
and CV is the only one of the two that still means anything. This course is
mostly in the second regime.

🚫 **The bootstrap for standard errors.** We use bootstrap *resampling* in Week 8,
as the engine inside bagging. We do not use it for inference on coefficients —
that is STAT 345 material and this course is about prediction.